# Part 2, Topic 3: Voltage Glitching to Dump Memory (MAIN)

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://www.ChipWhisperer.io). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *In the previous labs, we learned how voltage glitching can be used for a similar function as clock glitching. We also learned about how it has fewer limitations, but can be less reliable for certain target setups. It also changes a great deal based on the properties of the glitch circuit itself - even changing a wire can have a huge effect.*

*In this lab, we'll use what we learned in the last lab to again attack the vulnerable serial printing of the bootloader*

**LEARNING OUTCOMES:**

* Applying previous glitch settings to new firmware
* Checking for success and failure when glitching
* Understanding how compiler optimizations can cause devices to behave in strange ways

## The Situation

You should already know the situation from your previous attempts at glitching this bootloader (as well as what the flaw is). No need to do big long searches for parameters to try glitching at the beginning of the loop, just use values that worked well for the previous tutorial.

Be careful that you don't accidentally put the spot we're trying to glitch outside of `glitch_spots` - if you used a repeat > 1, the actual spot being glitched might be at the end or in the middle of the repeat!

Like with the clock glitching version of this lab, we'll be using SimpleSerial V2 to speed up glitching

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
SS_VER='SS_VER_2_1'

In [2]:
%%bash -s "$PLATFORM" "$SS_VER"
realpath ../../../firmware/mcu/bootloader-glitch
cd ../../../firmware/mcu/bootloader-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE -j SS_VER=$2

/Users/xcrbox/Desktop/2026-eCTF/chipwhisperer/firmware/mcu/bootloader-glitch
SS_VER set to SS_VER_2_1
arm-none-eabi-gcc (Arm GNU Toolchain 14.3.Rel1 (Build arm-14.174)) 14.3.1 20250623
Copyright (C) 2024 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWHUSKY 
.
.
.
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
.
.
.
.
.
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
    bootloader.c ...    simpleserial.c ...    decryption.c ...    .././hal//sam4s/uart.c ...    .././hal/hal.c ...    .././hal//sam4s/pmc.c ...    .././hal//sam4s/sam4s_hal.c ...    .././hal//sam4s/sysclk.c ...    .././hal//sam4s/pio.c ...    .././hal//sam4s/startup_sam4s.c ...    .././hal//sam4s/system_sam4s.c ...Done!
Done!
Done!
Done!
Done!
Done!
Done!
Done!
Done!
Done!
Done!
.
LI

In [3]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

/Users/xcrbox/Desktop/2026-eCTF/chipwhisperer/jupyter/.venv/lib/python3.10/site-packages/chipwhisperer/capture/trace/TraceWhisperer.py:31: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources # type: ignore


INFO: Found ChipWhisperer😍
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.glitch.phase_shift_steps           changed from 0                         to 4592              

In [4]:
fw_path = "../../../firmware/mcu/bootloader-glitch/bootloader-{}.hex".format(PLATFORM)

In [5]:
cw.program_target(scope, prog, fw_path)

Again, we're going to use a higher frequency on non-Husky ChipWhisperers. We'll also use the trigger length to get our `ext_offset` range:

In [6]:
#scope.clock.adc_src = "clkgen_x1"
scope.clock.clkgen_src = "system"
scope.adc.samples = 24000

def reboot_flush():
    reset_target(scope)
    #Flush garbage too
    target.flush()

reboot_flush()
scope.arm()
target.write("p516261276720736265747267206762206f686c207a76797821\n")

ret = scope.capture()
plot = cw.plot(scope.get_last_trace())
trig_count = scope.adc.trig_count
output = target.read(timeout=2)

print(list(output))
print(trig_count)
plot *=     cw.plot(scope.get_last_trace())

['r', '0', '\n', '\n', '\n', '\n', '\n', '\n']
60688


Like with the clock version of this lab, you'll want to inspect the power trace and glitch near the beginning and end of the loop.

In [7]:
scope.glitch.enabled = True
scope.glitch.clk_src = "pll"
scope.io.glitch_hp = True
scope.io.glitch_lp = False

scope.glitch.output = "glitch_only" # glitch_out = clk ^ glitch
scope.glitch.trigger_src = "ext_single" # glitch only after scope.arm() called

scope.adc.lo_gain_errors_disabled = True
scope.adc.clip_errors_disabled = True

scope.vglitch_setup('hp', default_setup=False)

def my_print(text):
    print(text)
    for ch in text:
        if (ord(ch) > 31 and ord(ch) < 127) or ch == "\n": 
            print(ch, end='')
        else:
            print("0x{:02X}".format(ord(ch)), end='')
        print("", end='')

In [8]:
scope.clock.adc_mul

4

In [9]:
scope.glitch.num_glitches

1

In [10]:
help(scope.clock)

Help on ChipWhispererHuskyClock in module chipwhisperer.capture.scopes.cwhardware.ChipWhispererHuskyClock object:

class ChipWhispererHuskyClock(chipwhisperer.common.utils.util.DisableNewAttr)
 |  ChipWhispererHuskyClock(oaiface: chipwhisperer.capture.scopes._OpenADCInterface.OpenADCInterface, fpga_clk_settings: chipwhisperer.capture.scopes._OpenADCInterface.ClockSettings, mmcm1, mmcm2, adc: chipwhisperer.capture.scopes.cwhardware.ChipWhispererHuskyMisc.ADS4128Settings)
 |  
 |  Method resolution order:
 |      ChipWhispererHuskyClock
 |      chipwhisperer.common.utils.util.DisableNewAttr
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __init__(self, oaiface: chipwhisperer.capture.scopes._OpenADCInterface.OpenADCInterface, fpga_clk_settings: chipwhisperer.capture.scopes._OpenADCInterface.ClockSettings, mmcm1, mmcm2, adc: chipwhisperer.capture.scopes.cwhardware.ChipWhispererHuskyMisc.ADS4128Settings)
 |      Initialize self.  See help(type(self)) for accurate signature.


In [11]:
help(scope.glitch)

Help on GlitchSettings in module chipwhisperer.capture.scopes.cwhardware.ChipWhispererGlitch object:

class GlitchSettings(chipwhisperer.common.utils.util.DisableNewAttr)
 |  GlitchSettings(cwglitch)
 |  
 |  Method resolution order:
 |      GlitchSettings
 |      chipwhisperer.common.utils.util.DisableNewAttr
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __init__(self, cwglitch)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  __repr__(self)
 |      Return repr(self).
 |  
 |  __str__(self)
 |      Return str(self).
 |  
 |  manualTrigger(self)
 |  
 |  manual_trigger(self) -> None
 |      Manually trigger the glitch output.
 |      
 |      This trigger is most useful in Manual trigger mode, where this is the
 |      only way to cause a glitch.
 |      
 |      Note that for ChipWhisperer-Husky, this method will only cause a glitch
 |      in manual mode, while on the Lite/Pro, this method will always insert a glitch.
 |  
 |  readStat

In [12]:
help(scope.adc)

Help on TriggerSettings in module chipwhisperer.capture.scopes._OpenADCInterface object:

class TriggerSettings(chipwhisperer.common.utils.util.DisableNewAttr)
 |  TriggerSettings(oaiface: chipwhisperer.capture.scopes._OpenADCInterface.OpenADCInterface)
 |  
 |  Method resolution order:
 |      TriggerSettings
 |      chipwhisperer.common.utils.util.DisableNewAttr
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __init__(self, oaiface: chipwhisperer.capture.scopes._OpenADCInterface.OpenADCInterface)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  __repr__(self)
 |      Return repr(self).
 |  
 |  __str__(self)
 |      Return str(self).
 |  
 |  clear_clip_errors(self)
 |      ADC clipping errors are sticky until manually cleared by calling this.
 |  
 |  disable_clip_and_lo_gain_errors(self, disabled)
 |  
 |  extTriggerPin(self)
 |  
 |  fifoOverflow(self)
 |  
 |  ----------------------------------------------------------------------
 | 

In [13]:
print(22000 / scope.clock.adc_mul)
print(23690 / scope.clock.adc_mul)

5500.0
5922.5


In [23]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset", "tries"])
gc.display_stats()

IntText(value=0, description='success count:', disabled=True)

IntText(value=0, description='reset count:', disabled=True)

IntText(value=0, description='normal count:', disabled=True)

FloatSlider(value=0.0, continuous_update=False, description='width setting:', disabled=True, max=10.0, readout…

FloatSlider(value=0.0, continuous_update=False, description='offset setting:', disabled=True, max=10.0, readou…

FloatSlider(value=0.0, continuous_update=False, description='ext_offset setting:', disabled=True, max=10.0, re…

FloatSlider(value=0.0, continuous_update=False, description='tries setting:', disabled=True, max=10.0, readout…

In [24]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_index="ext_offset", y_index="width")

Parameter name clashes for keys ['data']

:DynamicMap   []
   :Overlay
      .Points.I  :Points   [ext_offset,width]
      .Points.II :Points   [ext_offset,width]

Now the rest is up to you! 

In [27]:
#disable logging
cw.set_all_log_levels(cw.logging.CRITICAL)
scope.adc.timeout = 1.0
scope.glitch.num_glitches = 1

gc.set_range("width", 2400, 2600)
gc.set_range("offset", 2800, 3000)
gc.set_global_step(10)
#glitch_spots = list(range(0, trig_count, 1))

gc.set_range("ext_offset", 0, 2000)
gc.set_step("ext_offset", 50)
gc.set_range("tries", 0, 0) # change this if you want to glitch each spot multiple times
gc.set_step("tries", 1)

broken = False
for glitch_setting in gc.glitch_values():
    scope.glitch.width = glitch_setting[0]
    scope.glitch.offset = glitch_setting[1]
    scope.glitch.ext_offset = glitch_setting[2]
    
    while scope.adc.state:
        gc.add("reset")
        reboot_flush()
        time.sleep(1)

    target.flush()
    scope.arm()
    #target.write("p516261276720736265747267206762206f686c207a76797821\n")
    target.write("p000000000000000000000000000000000000000000000000000\n")
    ret = scope.capture()
    if ret:
        #print('Timeout - no trigger')
        gc.add("reset")
        output = list(target.read(timeout=2).encode('utf-8'))
        if len(output) != 0:
            print(f'result: "{output}"')
        reboot_flush()
        time.sleep(1)
        continue
        
    time.sleep(0.05)
    output = target.read(timeout=2)
    broken = False

    if len(output) != 0 and output.strip() != 'r0':
        print('result: ' + str({
            "ext_offset": scope.glitch.ext_offset,
            "offset": scope.glitch.offset,
            "width": scope.glitch.width,
            "output": bytearray(output.encode('utf-8'))
        }))

    for __ in range(500):
        num_char = target.in_waiting()
        if num_char:
            output = target.read(timeout=50)
            my_print(f'> {output}')
            broken = True
            continue
        break

    if broken:
        print('success: ' + str({
            "ext_offset": scope.glitch.ext_offset,
            "offset": scope.glitch.offset,
            "width": scope.glitch.width,
            "output": output.encode("utf-8")
        }))
        gc.add("success")
        break
    
    gc.add("normal")

#reenable logging
cw.set_all_log_levels(cw.logging.WARNING)

result: {'ext_offset': 150, 'offset': 2960, 'width': 2440, 'output': bytearray(b'3')}
result: {'ext_offset': 150, 'offset': 2990, 'width': 2440, 'output': bytearray(b'3')}
result: {'ext_offset': 150, 'offset': 3000, 'width': 2440, 'output': bytearray(b'3')}
result: {'ext_offset': 150, 'offset': 2930, 'width': 2460, 'output': bytearray(b'\xc3\xb3')}
result: {'ext_offset': 150, 'offset': 2950, 'width': 2460, 'output': bytearray(b'3')}
result: {'ext_offset': 150, 'offset': 2950, 'width': 2480, 'output': bytearray(b'3')}
result: {'ext_offset': 150, 'offset': 2910, 'width': 2490, 'output': bytearray(b'3')}


DFS/BFS Search

In [32]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import plotly.io as pio

pio.renderers.default = 'iframe'


results = gc.calc(ignore_params=[], sort="success_rate")
data = [
    {
        "width": results[index][0][0],
        "offset": results[index][0][1],
        "ext_offset": results[index][0][2],
        "success": results[index][1]['success'],
        "reset": results[index][1]['reset']
    } for index in range(len(results))
]

rgba_colors = ['rgba({},{},0,{:.2f})'.format(255 * val['reset'], 255 * val['success'], val['success'] + (0.5 * val['reset'])) for val in data]  # Example: fade near zero

go.Figure(go.Scatter3d(
    x=[i['ext_offset'] for i in data], y=[i['width'] for i in data], z=[i['offset'] for i in data],
    mode='markers',
    marker=dict(size=1, color=rgba_colors)
)).show()

The attack was on a hardened design therefore the glitch attack was not successful.

In [29]:
import json

with open("data/result_2_3.json", 'w') as file:
    json.dump(data, file)

### Without Device

In [ ]:
with open("data/result_2_3.json", 'r') as file:
    data = json.load(file)

In [33]:
scope.dis()
target.dis()